## 項目情報（修正前）

患者番号  
性別 (1: 男性, 2: 女性)  
年齢  
入院日  
検査日  
入院-検査期間(日)  
採取状況 (1: 入院時, 2: 病棟定期)  
VRE結果 (1: 陽性, 2: 陰性)  
過去のVRE検出歴 (1: あり, 2: なし)  
感染経路 (1: もちこみ（過去歴なし）, 2: もちこみ（過去歴あり）, 3: 院内新規感染, 4: 該当なし)  
入院時申込病名  
過去3ヶ月以内の当院入院歴 (0: 該当なし, 1: あり, 2: なし)  
過去3ヶ月以内の他院入院歴 (0: 該当なし, 1: あり, 2: なし)  
入院歴のある病院 (0: 該当なし, 1: 流行地域病院の入院歴あり, 2: 入院歴なし, 3: 非流行地域の入院歴あり)  
過去3ヶ月以内の介護施設入所歴 (0: 該当なし, 1: あり, 2: なし)  
尿Ba (1: あり, 2: なし)  
個室トイレ (1: あり, 2: なし)  
おむつ使用 (1: あり, 2: なし)  
過去3ヶ月以内の抗菌薬使用歴 (1: あり, 2: なし)  
経管栄養 (1: あり, 2: なし)  

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

# --- src ---
from pathlib import Path
import sys
sys.path.append(str(Path("../").resolve()))
from src.plotting import plot_roc_curve_cv, plot_shap_summary

# --- Raw Data ---
raw_data_path = "../data/raw/"

# --- Folder to save plots ---
results_path = "../results/CV_plot/"

if not os.path.exists(results_path):
    os.makedirs(results_path)


📌 1. 基本情報（そのまま利用）
| 修正前の項目名    | 修正後の扱い                 |
| ---------- | ---------------------- |
| 患者番号       | そのまま（ID扱い、学習には使わない）    |
| 年齢         | そのまま連続値                |
| 入院日        | 入院-検査期間を使用するため学習では使わない |
| 検査日        | 同上                     |
| 入院-検査期間(日) | そのまま使用                 |

📌 2. 二値化（1/0 に変換する項目）

臨床的には あり＝1 / なし＝0 に統一して SHAP の解釈性を最大化。

| 修正前の項目名                              | 修正後（変換内容）                   |
| ------------------------------------ | --------------------------- |
| 性別 (1: 男, 2: 女)                      | 男＝1 / 女＝0                   |
| 過去のVRE検出歴 (1:あり, 2:なし)               | あり＝1 / なし＝0                 |
| 過去3ヶ月以内の当院入院歴 (0:該当なし, 1:あり, 2:なし)   | **1=1、その他=0**（該当なし/なしは同じ扱い） |
| 過去3ヶ月以内の他院入院歴 (0:該当なし, 1:あり, 2:なし)   | 同上：あり=1、その他=0               |
| 過去3ヶ月以内の介護施設入所歴 (0:該当なし, 1:あり, 2:なし) | あり=1、その他=0                  |
| 尿Ba (1:あり, 2:なし)                     | あり=1 / なし=0                 |
| 個室トイレ (1:あり, 2:なし)                   | あり=1 / なし=0                 |
| おむつ使用 (1:あり, 2:なし)                   | あり=1 / なし=0                 |
| 過去3ヶ月以内の抗菌薬使用歴 (1:あり, 2:なし)          | あり=1 / なし=0                 |
| 経管栄養 (1:あり, 2:なし)                    | あり=1 / なし=0                 |

📌 3. 多カテゴリ → One-Hot Encoding にする項目

SHAP で間違った順序関係を作らないため、カテゴリは One-Hot 化。

| 修正前の項目名                                            | 修正後の扱い                                    |
| -------------------------------------------------- | ----------------------------------------- |
| 採取状況 (1:入院時, 2:病棟定期)                               | 0/1でも良いが **One-Hot** が最も解釈しやすい            |
| 感染経路 (1:もちこみ(過去歴なし), 2:もちこみ(あり), 3:院内新規感染, 4:該当なし) | **One-Hot に変換（4カテゴリ → 4列）**               |
| 入院時申込病名                                            | テキスト → カテゴリ → One-Hot または Target Encoding |
| 入院歴のある病院 (0〜3)                                     | **One-Hot（0,1,2,3 それぞれ別列へ）**              |

📌 4. 目的変数（VRE 結果 学習機作成時に変更？）

| 修正前                | 修正後         |
| ------------------ | ----------- |
| VRE結果 (1:陽性, 2:陰性) | 陽性＝1 / 陰性＝0 |


In [ ]:
import sys
import sklearn
import imblearn
import shap
import scipy

print("Python:", sys.version)
print("scikit-learn:", sklearn.__version__)
print("imbalanced-learn:", imblearn.__version__)
print("SHAP:", shap.__version__)
print("SciPy:", scipy.__version__)

In [ ]:
# =====================================
# 初期設定
# =====================================
seed = 42              # ランダムシード
sm_k = 5               # SMOTEの近傍数
n_tree = 500           # ランダムフォレストの木の数
test_size = 0.2        # テストデータの割合
pcr_unit_cost=171      # PCR検査の単価（円）

# =====================================
# データの前処理
# =====================================
df_nyuin = pd.read_excel(os.path.join(raw_data_path, "日赤VRE 20260203.xlsx"), sheet_name="入院時ASC全例")
df_byoto = pd.read_excel(os.path.join(raw_data_path, "日赤VRE 20260203.xlsx"), sheet_name="病棟ASC全例")


# column名変更
conv_dict = {
    'Pt No': 'patient_id',
    'Sex\n1. Male\n2. Female': 'sex',
    'Age': 'age',
    '入院日': 'admission_date',
    '検査日': 'test_date',
    '入院-検査期間(day)': 'days_from_admission_to_test',
    'Setting\n1. 入院時\n2. 病棟定期': 'sampling_setting',
    'VRE\n1. 陽性\n2. 陰性': 'vre_result',
    '過去のVRE検出歴\n1. あり\n2. なし': 'past_vre_history',
    '感染経路\n1. もちこみ (過去歴なし)\n2. もちこみ（過去歴あり）\n3. 院内新規感染\n4. 該当なし': 'infection_route',
    '入院時申し込み病名': 'admission_diagnosis',
    '過去三ヶ月以内の当院入院歴\n1. あり\n2. なし': 'recent_admission_this_hospital',
    '過去3ヶ月以内他院入院歴\n1. あり\n2. なし': 'recent_admission_other_hospital',
    '入院歴のある病院\n1. 流行地域病院の入院歴あり \n2. 入院歴なし\n3. 非流行地域の入院歴あり': 'hospitalization_region_history',
    '過去3ヶ月以内の介護施設入所歴\n1. あり\n2. なし': 'recent_care_facility_stay',
    '尿Ba\n1. あり\n2. なし': 'urinary_catheter',
    'Pトイレ\n1. あり\n2. なし': 'private_toilet',
    'おむつ\n1. あり\n2. なし': 'diaper_use',
    '過去3ヶ月以内の抗菌薬使用歴\n1. あり\n2. なし': 'recent_antibiotics_use',
    '経管栄養\n1. あり\n2. なし': 'tube_feeding'
}

df_nyuin.rename(columns=conv_dict, inplace=True)
df_nyuin["admission_diagnosis_id"]=df_nyuin["admission_diagnosis"].str.extract(r"^(\d+)\.").astype(int)
df_byoto.rename(columns=conv_dict, inplace=True)



In [ ]:
# admission_diagnosis_id と 元の診断内容の対応表を作成
df_admission_diag_map = (
    df_nyuin.loc[:, ["admission_diagnosis_id", "admission_diagnosis"]]
    .drop_duplicates()
    .sort_values("admission_diagnosis_id")
    .reset_index(drop=True)
)

df_admission_diag_map


In [ ]:
# -----------------------------
# 0/1 二値化する列
# -----------------------------

"""
以下について0(該当なし)を含むレコードを削除, またそのレコードを表示
    "recent_admission_this_hospital"
    "recent_admission_other_hospital"
    "hospitalization_region_history"
    "recent_care_facility_stay"
"""
print("削除された入院データ:")
print(df_nyuin[
    (df_nyuin["recent_admission_this_hospital"] == 0) |
    (df_nyuin["recent_admission_other_hospital"] == 0) |
    (df_nyuin["hospitalization_region_history"] == 0) |
    (df_nyuin["recent_care_facility_stay"] == 0)
])

df_nyuin = df_nyuin[
    (df_nyuin["recent_admission_this_hospital"] != 0) &
    (df_nyuin["recent_admission_other_hospital"] != 0) &
    (df_nyuin["hospitalization_region_history"] != 0) &
    (df_nyuin["recent_care_facility_stay"] != 0)
].copy()

# byotoデータには該当レコードはないため、そのまま

binary_map = {
    "sex": {1: 1, 2: 0},
    "past_vre_history": {1: 1, 2: 0},
    "recent_admission_this_hospital": {1: 1, 0: 0, 2: 0},
    "recent_admission_other_hospital": {1: 1, 0: 0, 2: 0},
    "recent_care_facility_stay": {1: 1, 0: 0, 2: 0},
    "urinary_catheter": {1: 1, 2: 0},
    "private_toilet": {1: 1, 2: 0},
    "diaper_use": {1: 1, 2: 0},
    "recent_antibiotics_use": {1: 1, 2: 0},
    "tube_feeding": {1: 1, 2: 0},
}

def apply_binary_mapping(df):
    for col, mp in binary_map.items():
        if col in df.columns:
            df[col] = df[col].map(mp).astype(int)
    return df


df_nyuin = apply_binary_mapping(df_nyuin)
df_byoto = apply_binary_mapping(df_byoto)


# -----------------------------
# One-Hot 化する列
# -----------------------------
one_hot_cols = [
    "sampling_setting",               # 1:入院時, 2:病棟定期
    "infection_route",                # 1〜4
    "hospitalization_region_history", # 0〜3
]

def apply_onehot(df, columns):
    df = df.copy()
    for col in columns:
        if col in df.columns:
            df = pd.get_dummies(df, columns=[col], prefix=col)
    return df


df_nyuin = apply_onehot(df_nyuin, one_hot_cols)
df_byoto = apply_onehot(df_byoto, one_hot_cols)


# -----------------------------
# admission_diagnosis_id はすでに数値化済み
# -----------------------------
df_nyuin["admission_diagnosis_id"] = df_nyuin["admission_diagnosis_id"].astype(int)

# =====================================
# 特徴量とターゲットの再設定
# =====================================

# 入院データ用の特徴量
feature_columns_nyuin = [
    'sex',
    'age',
    # 'days_from_admission_to_test',
    # 'admission_diagnosis_id',
    'recent_admission_this_hospital',
    'recent_admission_other_hospital',
    # 'hospitalization_region_history_0',
    'hospitalization_region_history_1',
    'hospitalization_region_history_2',
    'hospitalization_region_history_3',
    'recent_care_facility_stay',
    'urinary_catheter',
    'private_toilet',
    'diaper_use',
    'recent_antibiotics_use',
    'tube_feeding'
]

# # sampling_settingとinfection_routeをOne-Hotで追加（存在する場合のみ）
# feature_columns_nyuin += [c for c in df_nyuin.columns if c.startswith("sampling_setting_")]
# feature_columns_nyuin += [c for c in df_nyuin.columns if c.startswith("infection_route_")]

# 病棟データ用（存在する列だけ抽出）
feature_columns_byoto = [
    'sex',
    'age',
    'days_from_admission_to_test',
    'urinary_catheter',
    'private_toilet',
    'diaper_use',
    'recent_antibiotics_use',
    'tube_feeding'
]

# feature_columns_byoto += [c for c in df_byoto.columns if c.startswith("sampling_setting_")]
# feature_columns_byoto += [c for c in df_byoto.columns if c.startswith("infection_route_")]

# =====================================
# X, y の作成
# =====================================

lavel_column = 'vre_result'

X_nyuin = df_nyuin[feature_columns_nyuin].copy()
X_byoto = df_byoto[feature_columns_byoto].copy()

# ターゲット（VRE結果: 陽性1 / 陰性0）
y_nyuin = df_nyuin[lavel_column].apply(lambda x: 1 if x == 1 else 0)
y_byoto = df_byoto[lavel_column].apply(lambda x: 1 if x == 1 else 0)

print("入院データ X_nyuin shape:", X_nyuin.shape)
print("病棟データ X_byoto shape:", X_byoto.shape)

# 陽性率の確認
print("\n入院データ VRE陽性率:", y_nyuin.mean())
print("病棟データ VRE陽性率:", y_byoto.mean())

In [ ]:
# Aのデータフレームを df とする
df_tmp = df_nyuin.copy()

# =========================
# 1. 他院入院歴（地域）を1列に戻す
# =========================
def map_region_history(row):
    if row['hospitalization_region_history_1'] == True:
        return 'VRE流行地域'
    elif row['hospitalization_region_history_3'] == True:
        return 'VRE非流行地域'
    elif row['hospitalization_region_history_2'] == True:
        return '入院歴なし'
    else:
        return '不明'

df_tmp['他院入院歴（地域）'] = df_tmp.apply(map_region_history, axis=1)

# =========================
# 2. VRE結果をラベル化
# =========================
df_tmp['VRE結果'] = df_tmp['vre_result'].map({
    1: 'Positive',
    2: 'Negative'
})

# =========================
# 3. 件数集計
# =========================
count_table = pd.crosstab(df_tmp['他院入院歴（地域）'], df_tmp['VRE結果'])

# 行の順番を指定
order = ['VRE流行地域', 'VRE非流行地域', '入院歴なし']
count_table = count_table.reindex(order)

# Positive / Negative の総数
n_pos = (df_tmp['VRE結果'] == 'Positive').sum()
n_neg = (df_tmp['VRE結果'] == 'Negative').sum()

# =========================
# 4. 割合付きの表を作成
# =========================
result_table = pd.DataFrame(index=count_table.index)

result_table['Positive'] = count_table['Positive'].fillna(0).astype(int).apply(
    lambda x: f"{x} ({x / n_pos * 100:.1f} %)" if n_pos > 0 else f"{x} (0.0 %)"
)

result_table['Negative'] = count_table['Negative'].fillna(0).astype(int).apply(
    lambda x: f"{x} ({x / n_neg * 100:.1f} %)" if n_neg > 0 else f"{x} (0.0 %)"
)

# 表示
result_table

In [ ]:
class_counts = pd.Series(y_nyuin).value_counts(normalize=True)  # normalize=True で割合を出す

print("クラスの割合:")
print(class_counts)
print(f"陽性者:{sum(y_nyuin == 1)}, 陰性者:{sum(y_nyuin == 0)}")

In [ ]:
plot_label_dict = {
    'sex': 'Sex, Male',
    'age': 'Age [IQR]',
    'days_from_admission_to_test': 'Days from admission',
    'vre_result': 'VRE colonization',
    'past_vre_history': 'Prior VRE colonization',
    # 'admission_diagnosis_id': 'Diagnosis ID',

    'recent_admission_this_hospital': 'Prior admission to this hospital',
    'recent_admission_other_hospital': 'Prior admission to another hospital',

    # 'hospitalization_region_history_0': 'Region: none', # この列は存在しない
    # 'hospitalization_region_history_1': 'Prior admission region: high-prevalence region',
    # 'hospitalization_region_history_2': 'Prior admission region: no prior admission',
    # 'hospitalization_region_history_3': 'Prior admission region: low-prevalence region',
    'hospitalization_region_history_1': 'High-prevalence region',
    'hospitalization_region_history_2': 'No prior admission',
    'hospitalization_region_history_3': 'Low-prevalence region',

    'recent_care_facility_stay': 'Residence in a long-term care facility',
    'urinary_catheter': 'Urinary catheter use',
    'private_toilet': 'Bedside commodes use',
    'diaper_use': 'Diaper use',
    'recent_antibiotics_use': 'Antimicrobial exposure',
    'tube_feeding': 'Enteral feeding',

    'sampling_setting_1': 'Sampling: admission',
    'sampling_setting_2': 'Sampling: ward',

    'infection_route_1': 'Imported (no history)',
    'infection_route_2': 'Imported (with history)',
    'infection_route_3': 'Nosocomial',
    'infection_route_4': 'None',

    'shizuoka_vre_cases_1mo': 'VRE cases (prefecture)',
    'tobu_vre_cases_1mo': 'VRE cases (district)',
    'tobu_vre_screening_negative_1mo': 'Screening –',
    'tobu_vre_screening_positive_1mo': 'Screening +',
    'tobu_vre_screening_rate_1mo': 'VRE Positivity rate',

    'ABHR_consumption_1mo': 'ABHR'}


## evaluate_clinical_utility

In [ ]:
def evaluate_posthoc(y_true, y_prob, n_bins=5):

    # =========================
    # 1. Calibration
    # =========================
    prob_true, prob_pred = calibration_curve(
        y_true,
        y_prob,
        n_bins=n_bins,
        strategy="quantile"
    )

    brier = brier_score_loss(y_true, y_prob)

    fig_cal, ax1 = plt.subplots(figsize=(6,6))
    ax1.plot(prob_pred, prob_true, marker="o", linewidth=2)
    ax1.plot([0,1],[0,1],"--", linewidth=1)
    ax1.set_xlabel("Predicted probability")
    ax1.set_ylabel("Observed proportion")
    ax1.set_title(f"Calibration curve (Brier = {brier:.3f})")
    ax1.set_xlim(0,1)
    ax1.set_ylim(0,1)
    plt.tight_layout()

    # =========================
    # 2. Decision Curve Analysis
    # =========================
    thresholds = np.linspace(0.01, 0.2, 100)
    N = len(y_true)
    prevalence = np.mean(y_true)

    net_benefit_model = []
    net_benefit_all = []
    net_benefit_none = [0]*len(thresholds)

    for pt in thresholds:
        pred = (y_prob >= pt).astype(int)

        TP = np.sum((pred==1) & (y_true==1))
        FP = np.sum((pred==1) & (y_true==0))

        nb = (TP/N) - (FP/N)*(pt/(1-pt))
        net_benefit_model.append(nb)

        nb_all = prevalence - (1-prevalence)*(pt/(1-pt))
        net_benefit_all.append(nb_all)

    fig_dca, ax2 = plt.subplots(figsize=(6,6))
    ax2.plot(thresholds, net_benefit_model, linewidth=2, label="Model")
    ax2.plot(thresholds, net_benefit_all, linewidth=1, label="Treat all")
    ax2.plot(thresholds, net_benefit_none, linewidth=1, label="Treat none")
    ax2.set_xlabel("Threshold probability")
    ax2.set_ylabel("Net Benefit")
    ax2.legend()
    plt.tight_layout()

    # =========================
    # 3. Screening reduction
    # =========================
    reduction = {}
    for pt in [0.03, 0.05, 0.1]:
        screened = np.sum(y_prob >= pt)
        reduction[pt] = 1 - screened/N

    metrics = {
        "brier": float(brier),
        "prevalence": float(prevalence),
        "screening_reduction": reduction
    }

    return fig_cal, fig_dca, metrics

# データのプロット

In [ ]:
def plot_violin(data, column, labels):
    """
    dataframes: データテーブルのリスト
    column: 数値データの列名
    labels: 各データテーブルのラベル
    """
    
    mean = np.mean(data[column])
    std_dev = np.std(data[column])
    threshold_high = mean + 3 * std_dev
    outliers_number = data[column][data[column]>=threshold_high].count()
    #data = data[data[column]<threshold_high]
    fig,ax = plt.subplots(figsize=(12, 12))
    sns.violinplot(data=data, x=labels, y=column, palette="tab20")
    plt.title(f'{column}',size=36)
    plt.ylabel(column,size=24)
    plt.xlabel('Dataset').set_visible(False)
    plt.xticks(rotation=45,size=24)
    plt.yticks(size=24)
    plt.ylim([0,(np.nanpercentile(data[column], 100))*1.1])
    #ax.axhline(y=threshold_high,color='red',linestyle='--')
    ax.spines['top'].set_color('white')  # X軸
    ax.spines['right'].set_color('white')
    plt.rcParams['font.family'] = 'Arial'
    plt.tight_layout()
    #plt.savefig(path+'muti_data/Plot/violin_'+ column+'_202502_lim90per.png')
    plt.show()

In [ ]:
# plot_violin(X_nyuin,'age',y_nyuin)

In [ ]:
def plot_stacked_bar(data, column, labels):
    """
    dataframes: データテーブルのリスト
    column: カテゴリカルデータの列名
    labels: 各データテーブルのラベル
    """
    summary = pd.crosstab(data[column], labels)
    summary_percent = summary.div(summary.sum(axis=1), axis=0) * 100

    summary_percent.columns = ['negative', 'positive']
    summary_percent.plot(kind='bar', stacked=True, figsize=(12, 8), colormap='tab20')
    plt.title(f'{column}',size = 36)
    plt.ylabel('%',size = 24)
  
    plt.legend(bbox_to_anchor=(0.85, 1), loc='upper left',fontsize=24)
    plt.xticks(rotation=45,size=24)
    plt.yticks(size=24)
    plt.tight_layout()
    #plt.savefig("/Users/matsuuratakeru/Library/Mobile Documents/com~apple~CloudDocs/研究/fwd バイコマイシン/'+ column+'.png')
    plt.rcParams['font.family'] = 'Arial'
    plt.show()

In [ ]:
# nyuin_columns = X_nyuin.drop('age',axis=1).columns
# for i in nyuin_columns:
#     print(i)
#     plot_stacked_bar(X_nyuin, i, y_nyuin)

In [ ]:
def plot_bar(data, column, labels):
    """
    dataframes: データテーブルのリスト
    column: カテゴリカルデータの列名
    labels: 各データテーブルのラベル
    """
    fig,ax = plt.subplots(figsize=(12, 12))
    positive_counts = data[column].value_counts(normalize=True) * 100  # パーセンテージで
    print(positive_counts)
    # 3. 通常の棒グラフで表示
    positive_counts.plot(kind='bar', figsize=(12, 8), color='skyblue')
    
    plt.title(f'Bar Chart for {column}',size = 36)
    plt.ylabel('%',size = 24)
  
    plt.legend(bbox_to_anchor=(0.7, 1), loc='upper left',fontsize=24)
    plt.xticks(rotation=45,size=24)
    plt.yticks(size=24)
    plt.tight_layout()
    ax.spines['top'].set_color('white')  # X軸
    ax.spines['right'].set_color('white')
    plt.rcParams['font.family'] = 'Arial'
    #plt.savefig("/Users/matsuuratakeru/Library/Mobile Documents/com~apple~CloudDocs/研究/fwd バイコマイシン/'+ column+'.png')
    
    plt.show()

In [ ]:
# nyuin_columns = X_nyuin.drop('age',axis=1).columns
# y_nyuin[y_nyuin==1].index
# X_nyuin_only_positive = X_nyuin.iloc[y_nyuin[y_nyuin==1].index,:]
# for i in nyuin_columns:
#     print(i)
#     plot_bar(X_nyuin_only_positive, i, y_nyuin[y_nyuin==1])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator

# =========================
# 1) 入力（ここだけ変更すればOK）
# =========================
excel_path = "静岡県・東部保健所管内のデータ.xlsx"
sheet_name = "東部保健所管内スクリーニング件数・新規スクリーニング陽性数"

col_date = "Date"
col_neg  = "VREスクリーニング陰性数（人）"
col_pos  = "VREスクリーニング新規陽性数（人）"

# =========================
# 2) データ読み込み
# =========================
df = pd.read_excel(os.path.join(raw_data_path,excel_path), sheet_name=sheet_name).copy()

df[col_date] = pd.to_datetime(df[col_date])
df = df.sort_values(col_date)

neg = df[col_neg]
pos = df[col_pos]
rate = pos / (neg + pos) * 100

# =========================
# 3) 図の作成（投稿向け）
# =========================
fig, ax1 = plt.subplots(figsize=(11.5, 6.2))

# 色（直感的）
c_neg = "#B0B0B0"   # 陰性：グレー
c_pos = "#D62728"   # 陽性：赤
c_rate = "#1F77B4"  # 陽性率：青

# バー幅：月次なので日数で指定（細すぎ問題対策）
bar_width_days = 25

# 積み上げ棒（陰性＋新規陽性）
ax1.bar(df[col_date], neg, width=bar_width_days, color=c_neg,
        edgecolor="black", linewidth=0.6, label="VRE-negative screenings")
ax1.bar(df[col_date], pos, width=bar_width_days, bottom=neg, color=c_pos,
        edgecolor="black", linewidth=0.6, label="New VRE-positive cases")

ax1.set_ylabel("Number of screenings")
# ax1.set_xlabel("Month")

# 第2軸：陽性率（%）
ax2 = ax1.twinx()
ax2.plot(df[col_date], rate, color=c_rate, marker="o",
         linewidth=2.2, markersize=4.5, label="VRE positivity rate (%)")
ax2.set_ylabel("Positivity rate (%)")
ax2.yaxis.set_major_locator(MaxNLocator(nbins=6))

# タイトル
# ax1.set_title("Temporal trends in VRE screening results\nEastern Shizuoka district", pad=12)

# -------------------------
# X軸：月だけ表示（Jun/Jul...）、年は下に表示（Style B）
# ※ データがない月が出ないように軸範囲をデータに合わせて切る
# -------------------------
ax1.set_xlim(df[col_date].min() - pd.Timedelta(days=15),
             df[col_date].max() + pd.Timedelta(days=15))

ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b"))  # Jun, Jul, ...
plt.setp(ax1.get_xticklabels(), rotation=0, ha="center")

# 年の表示（年ごとの最初の月の位置に表示）
for y in sorted(df[col_date].dt.year.unique()):
    d0 = df.loc[df[col_date].dt.year == y, col_date].min()
    ax1.annotate(
        str(y),
        xy=(d0, 0),
        xycoords=("data", "axes fraction"),
        xytext=(0, -32),
        textcoords="offset points",
        ha="center",
        va="top",
        fontsize=11,
        fontweight="bold"
    )

# グリッド（薄め）
ax1.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.35)
ax1.set_axisbelow(True)

# 枠（スパイン）を黒で明確に
for ax in (ax1, ax2):
    for spine in ax.spines.values():
        spine.set_linewidth(1.0)
        spine.set_color("black")

# 凡例：下に横並び（3列）
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
fig.legend(
    h1 + h2, l1 + l2,
    loc="lower center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, -0.02)
)

# レイアウト調整（凡例ぶん下に余白）
plt.tight_layout(rect=[0, 0.04, 1, 1])

# =========================
# 4) 保存
# =========================

out_png_name = "Fig_regional_VRE_epidemiology_styleB_legendBottom.png"
out_png = os.path.join(results_path, out_png_name)
plt.savefig(out_png, dpi=600, bbox_inches="tight")
plt.show()

print("Saved:", out_png)


## 特徴量の相関

In [ ]:
# 特徴量の相関をプロット
def plot_correlation_heatmap(X, title):
    plt.figure(figsize=(12, 12))
    corr = X.corr()
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True, cbar_kws={"shrink": .8})
    plt.title(title, fontsize=24)
    plt.xticks(rotation=90, fontsize=16)
    plt.yticks(rotation=0, fontsize=16)
    plt.tight_layout()
    plt.show()  
    
plot_correlation_heatmap(X_nyuin, "Correlation Heatmap for X_nyuin")
plot_correlation_heatmap(X_byoto, "Correlation Heatmap for X_byoto")

# 入院時

In [ ]:
results_nyuin = {}

## 分類機作成

In [ ]:
fig_tag = "nyuin"
lavel_column = 'vre_result'

X_nyuin = df_nyuin[feature_columns_nyuin].copy()
# ターゲット（VRE結果: 陽性1 / 陰性0）
y_nyuin = df_nyuin[lavel_column].apply(lambda x: 1 if x == 1 else 0)

for model_type in ["rf"]:
    print(f"=== {model_type.upper()} ROC Curve ===")
    final_model,cv_metrics, final_thr, all_y_true, all_y_proba, figures, tprs, aucs = plot_roc_curve_cv(X_nyuin, y_nyuin, f"{model_type.upper()} CV ROC", seed=seed, model_type=model_type)
    figures["roc"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_roc.png"))
    figures["pr"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_pr.png"))
    figures["dca_zoom"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_dca_zoom.png"))
    figures["table"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_metrics_table.png"))
    figures["calib"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_calibration.png"))
    figures["pcr"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_pcr.png"))
    figures["pcr_target"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_pcr_target.png"))

    results_nyuin[fig_tag] = {
        "final_model": final_model,
        "cv_metrics": cv_metrics,
        "final_thr": final_thr,
        "all_y_true": all_y_true,
        "all_y_proba": all_y_proba,
        "figures": figures,
        "tprs": tprs,
        "aucs": aucs
    }

    shap_value, fig_shap = plot_shap_summary(X_nyuin, y_nyuin, f"{model_type.upper()} SHAP Summary", n_tree, seed, model_type, plot_label_dict=plot_label_dict)
    fig_shap.savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_shap.png"),
    dpi=300,
    bbox_inches="tight")
    
    


In [ ]:


def add_sensitivity_target_to_pcr_plot(
    all_y_true,
    all_y_proba,
    fig_pcr,
    target_sens=0.90,
    color="red",
    plt_show=True
):
    y_true = np.asarray(all_y_true)
    proba = np.asarray(all_y_proba)

    thresholds = np.linspace(0, 1, 1001)

    rows = []
    for t in thresholds:
        pred = (proba >= t).astype(int)

        tp = np.sum((y_true == 1) & (pred == 1))
        fn = np.sum((y_true == 1) & (pred == 0))
        tn = np.sum((y_true == 0) & (pred == 0))

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        pcr_reduction = (tn + fn) / len(y_true)  # 予測陰性割合

        rows.append((t, sensitivity, pcr_reduction))

    arr = np.array(rows)
    valid = arr[:, 1] >= target_sens

    # Sensitivity >= 0.9 を満たす中で、PCR削減率が最大の点
    best = arr[valid][np.argmax(arr[valid][:, 2])]

    threshold_prob = best[0]
    sensitivity_at_target = best[1]
    pcr_reduction_at_target = best[2]

    print(f"Target sensitivity        : {target_sens:.2f}")
    print(f"Probability threshold     : {threshold_prob:.3f}")
    print(f"Actual sensitivity        : {sensitivity_at_target:.3f}")
    print(f"PCR reduction rate        : {pcr_reduction_at_target:.3f}")

    ax = fig_pcr.axes[0]

    ax.axvline(target_sens, color=color, linestyle="--", lw=4)
    ax.axhline(pcr_reduction_at_target, color=color, linestyle="--", lw=4)
    ax.scatter(
        target_sens,
        pcr_reduction_at_target,
        color=color,
        s=180,
        zorder=5,
    )

    ax.text(
        target_sens + 0.02,
        pcr_reduction_at_target,
        f"Sensitivity={target_sens:.2f}\n"
        f"PCR reduction={pcr_reduction_at_target:.2f}\n"
        f"Threshold={threshold_prob:.3f}",
        color=color,
        fontsize=24,
        va="center",
    )
    if plt_show:
        plt.show()
    plt.close()
    

    return {
        "target_sensitivity": target_sens,
        "probability_threshold": threshold_prob,
        "actual_sensitivity": sensitivity_at_target,
        "pcr_reduction_rate": pcr_reduction_at_target,
        "fig": fig_pcr,
        "ax": ax,
    }

In [ ]:
result_09 = add_sensitivity_target_to_pcr_plot(
    all_y_true,
    all_y_proba,
    figures["pcr"],
    target_sens=0.90,
)



## 流行情報統合

In [ ]:
df_ryuko = pd.read_excel(os.path.join(raw_data_path, "静岡県・東部保健所管内のデータ.xlsx"), sheet_name="静岡県感染者数")
df_ryuko2 = pd.read_excel(os.path.join(raw_data_path, "静岡県・東部保健所管内のデータ.xlsx"), sheet_name="東部保健所管内スクリーニング件数・新規スクリーニング陽性数")

# Date列をkeyとして結合
df_ryuko = pd.merge(df_ryuko, df_ryuko2, on="Date", how="outer")

# column名変更
conv_dict = {
    '静岡県全VRE感染者数(人)': 'shizuoka_vre_cases',
    '東部保健所管内VRE感染者数（人）': 'tobu_vre_cases',
    'VREスクリーニング陰性数（人）': 'tobu_vre_screening_negative',
    'VREスクリーニング新規陽性数（人）': 'tobu_vre_screening_positive',
}

df_ryuko.rename(columns=conv_dict, inplace=True)
df_ryuko["tobu_vre_screening_rate"] = df_ryuko["tobu_vre_screening_positive"] / (df_ryuko["tobu_vre_screening_negative"] + df_ryuko["tobu_vre_screening_positive"])

In [ ]:
# --- 1. 入院日(admission_date)をdatetime型にして月初に丸める ---
df_nyuin['admission_date'] = pd.to_datetime(df_nyuin['admission_date'])
df_nyuin['month_start'] = df_nyuin['admission_date'].values.astype('datetime64[M]')
df_nyuin['month_start'] = pd.to_datetime(df_nyuin['month_start'])  # Timestamp型に戻す

# --- 2. df_ryukoの日付(Date)もdatetime型にし月初に丸める ---
df_ryuko['Date'] = pd.to_datetime(df_ryuko['Date'])
df_ryuko['Date'] = df_ryuko['Date'].values.astype('datetime64[M]')
df_ryuko['Date'] = pd.to_datetime(df_ryuko['Date'])

# --- 3. df_nyuinに1ヶ月前、2ヶ月前の月初日列を作る ---
df_nyuin['last_month_start'] = df_nyuin['month_start'] - pd.DateOffset(months=1)
df_nyuin['two_months_ago_start'] = df_nyuin['month_start'] - pd.DateOffset(months=2)

# --- 4. df_ryukoを1ヶ月前・2ヶ月前用にコピーし、suffixで列名を分ける ---
df_ryuko_1mo = df_ryuko.add_suffix('_1mo').rename(columns={'Date_1mo': 'Date_1mo'})
df_ryuko_2mo = df_ryuko.add_suffix('_2mo').rename(columns={'Date_2mo': 'Date_2mo'})

# --- 5. 1ヶ月前分をマージ ---
df_merged = pd.merge(df_nyuin, df_ryuko_1mo,
                     left_on='last_month_start', right_on='Date_1mo',
                     how='left')

# --- 6. の処理は最後にまとめるため削除 ---
# cols_1mo = [col for col in df_merged.columns if col.endswith('_1mo')]
# df_merged.drop(columns=cols_1mo, inplace=True)

# --- 7. 2ヶ月前分をマージ ---
df_merged = pd.merge(df_merged, df_ryuko_2mo,
                     left_on='two_months_ago_start', right_on='Date_2mo',
                     how='left')

# --- 9. 結果を元の変数に代入 ---
df_nyuin = df_merged

# 結果の確認
# print(df_nyuin.columns)

In [ ]:
features_1mo = [
    'shizuoka_vre_cases_1mo', 
    'tobu_vre_cases_1mo', 
    # 'tobu_vre_screening_negative_1mo', 
    # 'tobu_vre_screening_positive_1mo',
    'tobu_vre_screening_rate_1mo'
]

features_2mo = [
    'shizuoka_vre_cases_2mo', 
    'tobu_vre_cases_2mo', 
    # 'tobu_vre_screening_negative_2mo', 
    # 'tobu_vre_screening_positive_2mo',
    'tobu_vre_screening_rate_2mo'
]

df_nyuin_nan_removed_1mo = df_nyuin.dropna(subset=features_1mo)
# 削除した列を表示
print(f"removed {df_nyuin.shape[0] - df_nyuin_nan_removed_1mo.shape[0]} samples due to NaN in 1mo features")

df_nyuin_nan_removed_2mo = df_nyuin.dropna(subset=features_1mo + features_2mo)
print(f"After merging, dataset size: {df_nyuin_nan_removed_2mo.shape[0]} samples")
# 削除した列を表示
print(f"removed {df_nyuin.shape[0] - df_nyuin_nan_removed_2mo.shape[0]} samples due to NaN in 1mo + 2mo features")

fig_tag = "nyuin_1mo"
X_nyuin_nan_removed_1mo = df_nyuin_nan_removed_1mo[feature_columns_nyuin + features_1mo].copy()
y_nyuin_nan_removed_1mo = df_nyuin_nan_removed_1mo[lavel_column].apply(lambda x: 1 if x == 1 else 0)

for model_type in ["rf"]:
    print(f"=== {model_type.upper()} CV ROC ===")
    final_model,cv_metrics, final_thr, all_y_true, all_y_proba, figures, tprs, aucs = plot_roc_curve_cv(X_nyuin_nan_removed_1mo, y_nyuin_nan_removed_1mo, f"{model_type.upper()} CV ROC + 1mo", seed=seed, model_type=model_type)
    figures["roc"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_roc.png"))
    figures["pr"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_pr.png"))
    figures["dca_zoom"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_dca_zoom.png"))
    figures["table"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_metrics_table.png"))
    figures["calib"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_calibration.png"))
    figures["pcr"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_pcr.png"))
    figures["pcr_target"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_pcr_target.png"))
    
    
    results_nyuin[fig_tag] = {
        "final_model": final_model,
        "cv_metrics": cv_metrics,
        "final_thr": final_thr,
        "all_y_true": all_y_true,
        "all_y_proba": all_y_proba,
        "figures": figures,
        "tprs": tprs,
        "aucs": aucs
    }

    shap_value, fig_shap = plot_shap_summary(X_nyuin_nan_removed_1mo, y_nyuin_nan_removed_1mo, f"{model_type.upper()} SHAP Summary + 1mo", n_tree, seed, model_type=model_type, plot_label_dict=plot_label_dict)
    fig_shap.savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_shap.png"),
    dpi=300,
    bbox_inches="tight")

    

In [ ]:
from sklearn.metrics import roc_curve
from matplotlib.lines import Line2D

def plot_combined_roc(results_nyuin, colors, labels=None):

    FONT_TITLE  = 60
    FONT_LABEL  = 48
    FONT_TICK   = 40
    FONT_LEGEND = 32
    LW          = 5
    COLORS = [
        "#1D9E75", "#534AB7", "#D85A30",
        "#BA7517", "#185FA5", "#C03060",
    ]
    mean_fpr = np.linspace(0, 1, 100)

    fig, ax = plt.subplots(figsize=(12, 12))
    legend_handles = []

    for idx, (name, res) in enumerate(results_nyuin.items()):
        color      = colors[idx % len(colors)] if colors else COLORS[idx % len(COLORS)]
        label_name = (labels or {}).get(name, name)

        tprs     = np.array(res["tprs"])
        aucs     = np.array(res["aucs"])

        mean_tpr     = np.mean(tprs, axis=0)
        mean_tpr[-1] = 1.0
        std_tpr      = np.std(tprs, axis=0)
        mean_auc     = np.mean(aucs)
        std_auc      = np.std(aucs)

        # ax.fill_between(
        #     mean_fpr,
        #     np.maximum(mean_tpr - std_tpr, 0),
        #     np.minimum(mean_tpr + std_tpr, 1),
        #     color=color, alpha=0.10,
        # )
        ax.plot(mean_fpr, mean_tpr, color=color, lw=LW )
        legend_handles.append(
            Line2D([0], [0], color=color, lw=LW,
                   label=f"{label_name}  \n({mean_auc:.2f}±{std_auc:.2f})")
        )

    # ax.plot([0, 1], [0, 1], linestyle="--", lw=LW, color="black", alpha=0.5)
    # legend_handles.append(
    #     Line2D([0], [0], color="black", lw=LW, ls="--", label="Chance")
    # )

    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.set_aspect("equal")
    ax.set_title("", weight="bold", fontsize=FONT_TITLE)
    ax.set_xlabel("False Positive Rate", fontsize=FONT_LABEL)
    ax.set_ylabel("True Positive Rate",  fontsize=FONT_LABEL)
    ax.spines["right"].set_color("white")
    ax.spines["top"].set_color("white")
    ax.tick_params(labelsize=FONT_TICK)
    ax.legend(
        handles=legend_handles,
        loc=(0.5, 0.02), frameon=False,
        prop={"weight": "normal", "size": FONT_LEGEND},
    )
    plt.rcParams["font.family"] = "Arial"
    plt.tight_layout()
    plt.show()
    return fig

fig_tag = "nyuin"
fig_combined = plot_combined_roc(
    results_nyuin,
    colors=["#1D9E75", "#534AB7"],
    labels={"nyuin": "Base", "nyuin_1mo": "Base + Epi"},
)
fig_combined.savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_combined_roc.png"), dpi=300, bbox_inches="tight")

# 病棟

In [ ]:
results_byoto = {}

In [ ]:
fig_tag = "byoto"
for model_type in ["rf"]:
    print(f"=== {model_type.upper()} CV ROC (Byoto) ===")
    final_model,cv_metrics, final_thr, all_y_true, all_y_proba, figures, tprs, aucs  = plot_roc_curve_cv(X_byoto, y_byoto, f"{model_type.upper()} CV ROC", seed=seed, model_type=model_type)
    figures["roc"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_roc.png"))
    figures["pr"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_pr.png"))
    figures["dca_zoom"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_dca_zoom.png"))
    figures["table"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_metrics_table.png"))
    figures["calib"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_calibration.png"))
    figures["pcr"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_pcr.png"))
    figures["pcr_target"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_pcr_target.png"))
    
    
    results_byoto[fig_tag] = {
        "final_model": final_model,
        "cv_metrics": cv_metrics,
        "final_thr": final_thr,
        "all_y_true": all_y_true,
        "all_y_proba": all_y_proba,
        "figures": figures,
        "tprs": tprs,
        "aucs": aucs
    }

    shap_value, fig_shap = plot_shap_summary(X_byoto, y_byoto, f"{model_type.upper()} SHAP Summary", n_tree, seed, model_type, plot_label_dict=plot_label_dict)
    fig_shap.savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_shap.png"),
    dpi=300,
    bbox_inches="tight")

## アルコール使用状況統合

In [ ]:
df_ABHR = pd.read_excel(os.path.join(raw_data_path, "日赤アルコール使用量.xlsx"), sheet_name="Sheet1")

# column名変更
conv_dict = {
    '月': 'Date',
    'アルコール消費量（mL/患者/day）': 'ABHR_consumption',
}

df_ABHR.rename(columns=conv_dict, inplace=True)


In [ ]:

# test_dateをdatetime型に変換し、月初日に変換
df_byoto['test_date'] = pd.to_datetime(df_byoto['test_date'])
df_byoto['month_start'] = df_byoto['test_date'].values.astype('datetime64[M]')

# df_ABHRの日付もdatetimeにしておく
df_ABHR['Date'] = pd.to_datetime(df_ABHR['Date'])

# 月初でマージ
df_merged = pd.merge(df_byoto, df_ABHR, left_on='month_start', right_on='Date', how='left')

# 不要な列を整理（必要なら）
df_merged.drop(columns=['month_start', 'Date'], inplace=True)

df_byoto = df_merged

In [ ]:

# --- 1. 入院日(test_date)をdatetime型にして月初に丸める ---
df_byoto['test_date'] = pd.to_datetime(df_byoto['test_date'])
df_byoto['month_start'] = df_byoto['test_date'].values.astype('datetime64[M]')
df_byoto['month_start'] = pd.to_datetime(df_byoto['month_start'])  # Timestamp型に戻す

# --- 2. df_ABHRの日付(Date)もdatetime型にし月初に丸める ---
df_ABHR['Date'] = pd.to_datetime(df_ABHR['Date'])
df_ABHR['Date'] = df_ABHR['Date'].values.astype('datetime64[M]')
df_ABHR['Date'] = pd.to_datetime(df_ABHR['Date'])

# --- 3. df_byotoに1ヶ月前、2ヶ月前の月初日列を作る ---
df_byoto['last_month_start'] = df_byoto['month_start'] - pd.DateOffset(months=1)
df_byoto['two_months_ago_start'] = df_byoto['month_start'] - pd.DateOffset(months=2)

# --- 4. df_ABHRを1ヶ月前・2ヶ月前用にコピーし、suffixで列名を分ける ---
df_ABHR_current = df_ABHR.add_suffix('_current').rename(columns={'Date': 'Date'})
df_ABHR_1mo = df_ABHR.add_suffix('_1mo').rename(columns={'Date_1mo': 'Date_1mo'})
df_ABHR_2mo = df_ABHR.add_suffix('_2mo').rename(columns={'Date_2mo': 'Date_2mo'})

# --- 5. 1ヶ月前分をマージ ---
df_merged = pd.merge(df_byoto, df_ABHR_current,
                     left_on='month_start', right_on='Date_current',
                     how='left')

df_merged = pd.merge(df_merged, df_ABHR_1mo,
                     left_on='last_month_start', right_on='Date_1mo',
                     how='left')

df_merged = pd.merge(df_merged, df_ABHR_2mo,
                     left_on='two_months_ago_start', right_on='Date_2mo',
                     how='left')

# --- 9. 結果を元の変数に代入 ---
df_byoto = df_merged

# 結果の確認
print(df_merged.columns)



In [ ]:
# 陽性者数、陰性者数を確認
positive_count = df_byoto[df_byoto['vre_result'] == 1].shape[0]
negative_count = df_byoto[df_byoto['vre_result'] == 2].shape[0]
print(f"陽性者数: {positive_count}, 陰性者数: {negative_count}")

In [ ]:
# 特徴量にアルコール使用状況を追加
X_byoto = df_byoto[feature_columns_byoto + ['ABHR_consumption_1mo']]
fig_tag = "byoto_ABHR_1mo"
for model_type in ["rf"]:
    print(f"=== {model_type.upper()} CV ROC ===")
    final_model,cv_metrics, final_thr, all_y_true, all_y_proba, figures, tprs, aucs  = plot_roc_curve_cv(X_byoto, y_byoto, f"{model_type.upper()} CV ROC + 1mo", seed=seed, model_type=model_type)
    figures["roc"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_roc.png"))
    figures["pr"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_pr.png"))
    figures["dca_zoom"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_dca_zoom.png"))
    figures["table"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_metrics_table.png"))
    figures["calib"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_calibration.png"))
    figures["pcr"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_pcr.png"))
    figures["pcr_target"].savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_pcr_target.png"))
    
    
    results_byoto[fig_tag] = {
        "final_model": final_model,
        "cv_metrics": cv_metrics,
        "final_thr": final_thr,
        "all_y_true": all_y_true,
        "all_y_proba": all_y_proba,
        "figures": figures,
        "tprs": tprs,
        "aucs": aucs
    }
    
    shap_value, fig_shap = plot_shap_summary(X_byoto, y_byoto, f"{model_type.upper()} SHAP Summary", n_tree, seed, model_type, plot_label_dict=plot_label_dict)
    fig_shap.savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_shap.png"),     dpi=300,     bbox_inches="tight")


In [ ]:
fig_tag = "byoto"
fig_combined = plot_combined_roc(
    results_byoto,
    colors=[
        "#D85A30","#C03060",],
    labels={"byoto": "Base", "byoto_ABHR_1mo": "Base + ABHR"},
)
fig_combined.savefig(os.path.join(results_path, f"{fig_tag}_{model_type}_combined_roc.png"), dpi=300, bbox_inches="tight")